# Step 3 &mdash; Multi-Label Probe-Level Split

**Goal:** assign each probe (and all of its tiles) to one of train / val / test such that:

1. **No leakage** &mdash; all tiles from a probe stay in the same fold.
2. **Rare classes survive** &mdash; every class appears in val (and ideally test) whenever the data allows.
3. **Multi-label aware** &mdash; some probes carry up to 3 labels; the split must respect co-occurrence.

## Why a custom greedy split?

Generic stratified split assumes single-label samples. `skmultilearn`'s `IterativeStratification` works but adds a heavy dependency for one function. A small custom **rare-class-first greedy** algorithm gives the same guarantees with no extra deps and is easy to audit.

| Variant | Ratio | Rationale |
|---------|-------|-----------|
| CNN  | 70 / 20 / 10 | CNN baselines are sample-stable; 70% train is enough |
| Swin | 75 / 15 / 10 | Swin is more data-hungry; +5% train for transformer signal |

## Scan label folders &rarr; probe label sets

Folder name = class label. The same image (probe) can live in multiple class folders, which is exactly how the multi-label nature of the dataset is encoded.

In [ ]:
from collections import defaultdict
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def get_probe_id(filename):
    return Path(filename).stem.replace("_STD", "").replace("_ROI", "")

def scan_probes(roi_root):
    probe_labels = defaultdict(set)
    probe_files = {}
    classes = set()
    for cls_dir in Path(roi_root).iterdir():
        if not cls_dir.is_dir():
            continue
        classes.add(cls_dir.name)
        for p in cls_dir.rglob("*"):
            if p.suffix.lower() in IMG_EXTS:
                pid = get_probe_id(p.name)
                probe_labels[pid].add(cls_dir.name)
                probe_files.setdefault(pid, p)
    return probe_labels, probe_files, sorted(classes)

## Rare-class-first greedy split

1. Compute global class frequencies, sort classes from rarest to most common.
2. For each class (rare first), pull probes that contain that class into VAL (and TEST) until the per-class minimum is met or the quota is full. Prefer probes that carry the most labels (most "efficient" per pick).
3. Fill remaining VAL/TEST quota randomly.
4. Everything else goes to TRAIN.
5. Tiny rounding corrections at the end.

In [ ]:
import numpy as np

def multilabel_greedy_split(probe_labels, classes,
                            train_ratio=0.75, val_ratio=0.15,
                            seed=7, min_val=1, min_test=0):
    rng = np.random.default_rng(seed)
    ids = list(probe_labels.keys())
    rng.shuffle(ids)

    n = len(ids)
    n_train = round(n * train_ratio)
    n_val   = round(n * val_ratio)
    n_test  = n - n_train - n_val

    train, val, test = set(), set(), set()
    unassigned = set(ids)

    freq = {c: sum(1 for p in ids if c in probe_labels[p]) for c in classes}
    rare_first = sorted(classes, key=lambda c: freq[c])

    def count_label(split, c): return sum(1 for p in split if c in probe_labels[p])
    def pick(c):
        cands = [p for p in unassigned if c in probe_labels[p]]
        return max(cands, key=lambda p: len(probe_labels[p])) if cands else None

    # (1) Ensure minimum class coverage in VAL / TEST
    for c in rare_first:
        while count_label(val, c) < min_val and len(val) < n_val:
            p = pick(c)
            if p is None: break
            val.add(p); unassigned.remove(p)
        while count_label(test, c) < min_test and len(test) < n_test:
            p = pick(c)
            if p is None: break
            test.add(p); unassigned.remove(p)

    # (2) Fill remaining VAL / TEST randomly
    rest = list(unassigned); rng.shuffle(rest)
    for p in rest:
        if   len(val)  < n_val:  val.add(p);  unassigned.discard(p)
        elif len(test) < n_test: test.add(p); unassigned.discard(p)
        else: break

    # (3) Rest -> TRAIN
    train.update(unassigned)
    return list(train), list(val), list(test)

## Emit the CSV manifest

One row per **tile**, with metadata + a multi-hot vector across all classes. Downstream training code (CNN baselines, SwinV2) reads this single CSV.

In [ ]:
import pandas as pd

def write_manifest(probe_labels, probe_files, classes,
                   train_ids, val_ids, test_ids,
                   img_tiles_dir, img_global_dir, out_csv):
    split_map = {p: "train" for p in train_ids}
    split_map.update({p: "val"  for p in val_ids})
    split_map.update({p: "test" for p in test_ids})

    rows = []
    img_tiles_dir = Path(img_tiles_dir)
    for pid, file_path in probe_files.items():
        base = Path(file_path).stem
        global_fname = f"{pid}_global.jpg"
        label_vec = {c: int(c in probe_labels[pid]) for c in classes}

        # one row per tile that was actually written in Step 2
        for tile_path in sorted(img_tiles_dir.glob(f"{pid}_tile*.jpg")):
            idx = int(tile_path.stem.split("_tile")[-1])
            rows.append({
                "filename_tile":   tile_path.name,
                "filename_global": global_fname,
                "split":           split_map[pid],
                "probe_id":        pid,
                "tile_index":      idx,
                **label_vec,
            })

    cols = ["filename_tile", "filename_global", "split", "probe_id", "tile_index"] + classes
    pd.DataFrame(rows)[cols].to_csv(out_csv, index=False)
    print(f"Manifest -> {out_csv}")

## Sanity check &mdash; per-class positive-rate diff

After the split, the positive rate of every label should be similar across train / val / test. Large drift (&gt; 5%) suggests the split needs re-seeding or stricter `min_val` / `min_test`.

In [ ]:
def split_distribution(csv_path):
    df = pd.read_csv(csv_path)
    meta = {"filename_tile", "filename_global", "split", "probe_id", "tile_index"}
    label_cols = [c for c in df.columns if c not in meta]
    probe_df = df.groupby(["probe_id", "split"], as_index=False)[label_cols].max()
    rates = probe_df.groupby("split")[label_cols].mean().T
    rates.columns = [f"rate_{c}" for c in rates.columns]
    rates["diff_val_minus_train"] = rates.get("rate_val", 0) - rates.get("rate_train", 0)
    return rates.sort_values("diff_val_minus_train", ascending=False)

# split_distribution("./outputs/swin/02_DualStream/dataset_dualstream.csv")

## Result &mdash; what came out of the split

| Variant | Train | Val | Test |
|---------|-------|-----|------|
| CNN  (70/20/10) | 771 probes | 221 | 108 |
| Swin (75/15/10) | 825 probes | 165 | 110 |

Both variants ship a single `dataset_dualstream.csv` that the training scripts read with `df[df.split == "train"]` etc.